# 03 - 模型对比

本笔记本用于对比不同机器学习模型在水印检测任务上的性能表现。

## 1. 导入必要的库

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import xgboost as xgb
import lightgbm as lgb

from src.data_loader import DataLoader
from src.feature_extractor import FeatureExtractor

%matplotlib inline

## 2. 准备数据

In [ ]:
# 加载数据
loader = DataLoader(time_column='timestamp')
df = loader.load('../data/sample/sample_data.csv')

# 提取特征
extractor = FeatureExtractor(
    window_size=5,
    step_size=1,
    feature_groups=['statistical', 'time_domain', 'watermark_specific']
)

features_df = extractor.extract(df, target_column='close')
feature_cols = [c for c in features_df.columns if c not in ['window_start', 'window_end']]

X = features_df[feature_cols].values

# 获取标签
y = []
for _, row in features_df.iterrows():
    window_data = df.loc[row['window_start']:row['window_end']]
    if 'watermark_label' in window_data.columns:
        label = window_data['watermark_label'].mode()[0]
    else:
        label = 0
    y.append(label)
y = np.array(y)

print(f"特征矩阵形状: {X.shape}")
print(f"标签分布: {np.bincount(y)}")

## 3. 定义要对比的模型

In [ ]:
# 定义模型
models = {
    'XGBoost': xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1,
        verbosity=0
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42
    ),
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        random_state=42,
        n_jobs=-1
    ),
    'SVM': SVC(
        kernel='rbf',
        C=1.0,
        probability=True,
        random_state=42
    )
}

print(f"将对比 {len(models)} 个模型")
for name in models.keys():
    print(f"  - {name}")

## 4. 交叉验证对比

In [ ]:
# 定义评估指标
metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

# 存储结果
results = []

# 交叉验证
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for name, model in models.items():
    print(f"\n评估模型: {name}")
    
    model_results = {'model': name}
    
    # 训练时间
    start_time = time.time()
    
    # 计算各指标
    for metric in metrics:
        scores = cross_val_score(model, X, y, cv=cv, scoring=metric)
        model_results[f'{metric}_mean'] = scores.mean()
        model_results[f'{metric}_std'] = scores.std()
        print(f"  {metric:12s}: {scores.mean():.4f} (+/- {scores.std()*2:.4f})")
    
    # 推理时间估算
    model.fit(X, y)
    inference_start = time.time()
    _ = model.predict(X)
    inference_time = (time.time() - inference_start) / len(X) * 1000  # ms per sample
    
    model_results['inference_time_ms'] = inference_time
    model_results['train_time_s'] = time.time() - start_time
    
    results.append(model_results)

results_df = pd.DataFrame(results)

## 5. 结果可视化

In [ ]:
# 创建对比图表
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

plot_metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

for idx, metric in enumerate(plot_metrics):
    ax = axes[idx]
    
    means = results_df[f'{metric}_mean']
    stds = results_df[f'{metric}_std']
    
    bars = ax.bar(results_df['model'], means, yerr=stds, capsize=5, alpha=0.7)
    ax.set_ylabel(metric.capitalize())
    ax.set_ylim(0, 1.1)
    ax.set_title(f'{metric.capitalize()} Comparison')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)
    
    # 在柱子上标注数值
    for bar, mean in zip(bars, means):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{mean:.3f}',
                ha='center', va='bottom', fontsize=8)

# 推理时间对比
axes[5].bar(results_df['model'], results_df['inference_time_ms'], alpha=0.7, color='coral')
axes[5].set_ylabel('Inference Time (ms)')
axes[5].set_title('Inference Time Comparison')
axes[5].tick_params(axis='x', rotation=45)
axes[5].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 综合性能雷达图
from math import pi

categories = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']
N = len(categories)

# 计算角度
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

colors = plt.cm.tab10(np.linspace(0, 1, len(models)))

for idx, (_, row) in enumerate(results_df.iterrows()):
    values = [
        row['accuracy_mean'],
        row['precision_mean'],
        row['recall_mean'],
        row['f1_mean'],
        row['roc_auc_mean']
    ]
    values += values[:1]
    
    ax.plot(angles, values, 'o-', linewidth=2, label=row['model'], color=colors[idx])
    ax.fill(angles, values, alpha=0.1, color=colors[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)
ax.set_ylim(0, 1)
ax.set_title('Model Performance Radar Chart', size=16, y=1.08)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
ax.grid(True)

plt.tight_layout()
plt.show()

## 6. 详细结果表格

In [ ]:
# 创建汇总表格
summary_df = pd.DataFrame({
    'Model': results_df['model'],
    'Accuracy': results_df['accuracy_mean'].apply(lambda x: f"{x:.4f}"),
    'Precision': results_df['precision_mean'].apply(lambda x: f"{x:.4f}"),
    'Recall': results_df['recall_mean'].apply(lambda x: f"{x:.4f}"),
    'F1 Score': results_df['f1_mean'].apply(lambda x: f"{x:.4f}"),
    'AUC': results_df['roc_auc_mean'].apply(lambda x: f"{x:.4f}"),
    'Inference (ms)': results_df['inference_time_ms'].apply(lambda x: f"{x:.2f}"),
})

print("模型性能对比汇总:")
print("=" * 100)
print(summary_df.to_string(index=False))
print("=" * 100)

## 7. 最佳模型深度分析

In [ ]:
# 找出最佳模型（按F1分数）
best_model_idx = results_df['f1_mean'].idxmax()
best_model_name = results_df.loc[best_model_idx, 'model']
best_model = models[best_model_name]

print(f"最佳模型: {best_model_name}")
print(f"F1 Score: {results_df.loc[best_model_idx, 'f1_mean']:.4f}")

In [ ]:
from sklearn.model_selection import train_test_split

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 训练最佳模型
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)

# 分类报告
print("\n分类报告:")
print(classification_report(y_test, y_pred, target_names=['No Watermark', 'Watermark']))

In [ ]:
# 混淆矩阵
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Watermark', 'Watermark'],
            yticklabels=['No Watermark', 'Watermark'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.show()

In [ ]:
# ROC曲线
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_test, y_proba[:, 1])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'ROC Curve - {best_model_name}')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 特征重要性（如果是基于树的模型）
if hasattr(best_model, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    plt.figure(figsize=(10, 8))
    plt.barh(range(len(importance_df)), importance_df['importance'])
    plt.yticks(range(len(importance_df)), importance_df['feature'])
    plt.xlabel('Importance')
    plt.title(f'Feature Importance - {best_model_name}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print("\nTop 10 重要特征:")
    print(importance_df.head(10))

## 8. 总结与建议

基于以上实验，我们可以得出以下结论：

1. **最佳模型**: 根据F1分数和其他指标，推荐使用的模型
2. **性能对比**: 各模型在不同指标上的表现差异
3. **推理速度**: 模型的推理时间对比，对于实时检测的重要性
4. **模型选择建议**: 根据具体应用场景（精度优先还是速度优先）给出建议

这些结论将用于优化配置文件中的模型选择。